### Connection Mongo DB

In [ ]:
pip install pymongo

In [ ]:
import pymongo

In [ ]:
from pymongo import MongoClient

conn = MongoClient("mongodb://localhost:27017")

db = conn["university"]

In [ ]:
print(db.list_collection_names())

### Extract Data

In [ ]:
import pandas as pd

In [ ]:
dfstudents = pd.DataFrame(list(db.students.find({})))
dfstudents

In [ ]:
dfattendances = pd.DataFrame(list(db.attendances.find({})))
dfattendances

In [ ]:
dfgrades = pd.DataFrame(list(db.grades.find({})))
dfgrades

### Transform Data

In [81]:
dfstudents.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 10 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   _id         100 non-null    object 
 1   student_id  100 non-null    int64  
 2   name        100 non-null    str    
 3   age         100 non-null    Int64  
 4   city        100 non-null    str    
 5   gpa         100 non-null    float64
 6   email       100 non-null    str    
 7   phone       100 non-null    str    
 8   gender      100 non-null    str    
 9   level       100 non-null    Int64  
dtypes: Int64(2), float64(1), int64(1), object(1), str(5)
memory usage: 11.7+ KB


In [ ]:
dfstudents.isnull().sum()

In [ ]:
dfstudents['email'] = dfstudents['email'].fillna('UNKNOWN')
dfstudents['phone'] = dfstudents['phone'].fillna('UNKNOWN')
dfstudents['gender'] = dfstudents['gender'].fillna('UNKNOWN')
dfstudents['level'] = dfstudents['level'].fillna(dfstudents['level'].mode()[0])
dfstudents['age'] = dfstudents['age'].fillna(dfstudents['age'].mode()[0])


In [ ]:
dfstudents['age'] = dfstudents['age'].astype('Int64')

In [ ]:
dfstudents['level'] = dfstudents['level'].astype('Int64')

In [ ]:
dfstudents['city'].value_counts()

In [ ]:
dfstudents.isnull().sum()

In [82]:
dfattendances.info()

<class 'pandas.DataFrame'>
RangeIndex: 104 entries, 0 to 103
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   _id         104 non-null    object        
 1   student_id  104 non-null    int64         
 2   date        104 non-null    datetime64[us]
 3   status      104 non-null    str           
 4   note        104 non-null    str           
dtypes: datetime64[us](1), int64(1), object(1), str(2)
memory usage: 5.6+ KB


In [ ]:
dfattendances['date'] = pd.to_datetime(dfattendances['date'], format='%Y-%m-%d')

In [ ]:
dfattendances.isnull().sum()

In [ ]:
dfattendances['status'].value_counts()

In [ ]:
dfattendances['status'] = dfattendances['status'].replace({'present':'Present'}, inplace=True)

In [ ]:
dfattendances['status'].value_counts()

In [ ]:
dfattendances['status'] = dfattendances['status'].fillna(dfattendances['status'].mode()[0])

In [ ]:
dfattendances['note'] = dfattendances['note'].fillna('No note')

In [ ]:
dfattendances['note'].value_counts()

In [ ]:
dfattendances.isnull().sum()

In [ ]:
dfattendances.info()

In [83]:
dfgrades.info()

<class 'pandas.DataFrame'>
RangeIndex: 105 entries, 0 to 104
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   _id         105 non-null    object
 1   student_id  105 non-null    int64 
 2   math        105 non-null    Int64 
 3   python      105 non-null    int64 
 4   note        105 non-null    str   
dtypes: Int64(1), int64(2), object(1), str(1)
memory usage: 5.1+ KB


In [ ]:
dfgrades['math'] = dfgrades['math'].astype('Int64')

In [ ]:
dfgrades.isnull().sum()

In [ ]:
dfgrades['note'] = dfgrades['note'].fillna('No note')

In [ ]:
dfgrades.isnull().sum()

### Load Data

In [97]:
from sqlalchemy import create_engine

engine = create_engine(
    "mssql+pyodbc://DESKTOP-JKQGEOC\\SQLSERVER2025/universityDWH"
    "?trusted_connection=yes"
    "&driver=ODBC+Driver+17+for+SQL+Server"
    "&TrustServerCertificate=yes"
)

In [84]:

math_df =dfgrades[['student_id', 'math']].copy()
math_df = math_df.rename(columns={'math': 'grade'})
math_df['subject'] = 'math'

python_df = dfgrades[['student_id', 'python']].copy()
python_df = python_df.rename(columns={'python': 'grade'})
python_df['subject'] = 'python'

grades = pd.concat(
    [math_df, python_df],
    ignore_index=True
)

print(grades.head())

   student_id  grade subject
0           1     90    math
1           2     80    math
2           3     84    math
3           4     75    math
4           5     70    math


In [85]:
StudentPerformance = pd.merge(dfstudents, grades, on='student_id', how='left')
StudentPerformance = pd.merge(StudentPerformance, dfattendances, on='student_id', how='left')

In [87]:
StudentPerformance.columns

Index(['_id_x', 'student_id', 'name', 'age', 'city', 'gpa', 'email', 'phone',
       'gender', 'level', 'grade', 'subject', '_id_y', 'date', 'status',
       'note'],
      dtype='str')

In [ ]:
StudentPerformance = StudentPerformance[['student_id', 'name','age', 'city', 'subject' ,'grade', 'date', 'status']]

In [89]:
StudentPerformance = StudentPerformance.rename(columns={'name': 'student_name', 'date':'attendance_date', 'status':'attendance_status'})

In [93]:
StudentPerformance.columns

Index(['student_id', 'student_name', 'age', 'city', 'grade', 'subject',
       'attendance_date', 'attendance_status'],
      dtype='str')

In [94]:
StudentPerformance.head(5)

,student_id,student_name,age,city,grade,subject,attendance_date,attendance_status
0,1,Mohamed,22,Cairo,90,math,2026-10-01,Present
1,1,Mohamed,22,Cairo,87,python,2026-10-01,Present
2,2,Ahmed,23,Giza,80,math,2026-10-01,Absent
3,2,Ahmed,23,Giza,76,python,2026-10-01,Absent
4,3,Mai,21,Alexandria,84,math,2026-10-01,Present


In [98]:
StudentPerformance.to_sql('StudentPerformance', con=engine, if_exists='append', index=False)

200